In [ ]:
pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 26.9 MB/s eta 0:00:00


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Parameters and indices
n = 5  # Number of points
K = 3  # Number of products

np.random.seed(0)

# Distance cost matrix
C_t = 3 * np.random.rand(n, n)

# Initial shelf life, required shelf life, and quality reduction factors
SL_0 = 700 * np.random.rand(K)  # Initial shelf life
SL_r = 100 * np.random.rand(K)  # Required shelf life at destination
Q = np.random.uniform(0.5, 1, K)  # Quality reduction factor

# Priority customers
P = np.random.choice([0, 1], size=n, p=[.7, .3])
alpha = 7

In [ ]:
# Uncertainty sets for R (delays) and T (travel times)
R_min = 1.5 * np.random.rand(n)  # Lower bound for R
R_max = 2.5 * np.random.rand(n)  # Upper bound for R
T_min = 4 * np.random.rand(n, n)  # Lower bound for T
T_max = 6 * np.random.rand(n, n)  # Upper bound for T

In [ ]:
model = gp.Model('Robust Perishable Produce Transportation')


Restricted license - for non-production use only - expires 2025-11-24


In [ ]:
# Variables
x = model.addVars(n, n, vtype=GRB.BINARY)  # 1 if arc i-j is part of the route
u = model.addVars(n, lb=0, ub=float('inf'), vtype=GRB.CONTINUOUS)  # Auxiliary variable for subtour elimination


In [ ]:
# Constraints

# Each node is entered once
model.addConstrs((gp.quicksum(x[i, j] for i in range(n)) == 1 for j in range(n)), name='const1')

# No self-loops
model.addConstrs((x[i, j] == 0 for i in range(n) for j in range(n) if i == j), name='const2')

# Each node is exited once
model.addConstrs((gp.quicksum(x[i, j] for j in range(n)) == 1 for i in range(n)), name='const3')

# MTZ Constraints (Subtour elimination)
model.addConstrs((u[i] - u[j] + n * x[i, j] <= n for i in range(n) for j in range(n) if i != j), name='const4')

# Robust in-transit time constraint (worst-case)
for k in range(K):
    model.addConstr(
        gp.quicksum((R_max[i] + T_max[i, j]) * x[i, j] for i in range(n) for j in range(n)) <= Q[k] * SL_0[k] - SL_r[k],
        name=f'robust_time_constraint_k_{k}'
    )

In [ ]:
# Objective function: Minimize cost minus priority term
cost_term = gp.quicksum(C_t[i, j] * x[i, j] for i in range(n) for j in range(n))
priority_term = gp.quicksum((P[i] + P[j]) * alpha * x[i, j] for i in range(n) for j in range(n))

# Set the objective (minimize cost minus priority term)
model.setObjective(cost_term - priority_term, GRB.MINIMIZE)

In [ ]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 38 rows, 30 columns and 190 nonzeros
Model fingerprint: 0x667e5d40
Variable types: 5 continuous, 25 integer (25 binary)
Coefficient statistics:
  Matrix range     [9e-01, 6e+00]
  Objective range  [6e-02, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]
Found heuristic solution: objective -2.4697255
Presolve removed 8 rows and 5 columns
Presolve time: 0.00s
Presolved: 30 rows, 25 columns, 100 nonzeros
Variable types: 5 continuous, 20 integer (20 binary)

Root relaxation: objective -8.769467e+00, 8 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0       

In [ ]:
# Display results
if model.status == GRB.OPTIMAL:
    print(f"Optimal objective value: {model.objVal}")

    for var in model.getVars():
        if var.X > 0:
            print(f"{var.VarName}: {var.X}")
else:
    print(f"Optimization was unsuccessful. Status code: {model.status}")

Optimal objective value: -8.769466658135908
C3: 1.0
C5: 1.0
C14: 1.0
C16: 1.0
C22: 1.0
